<a href="https://colab.research.google.com/github/beginnerkuri/pakistan-womens-employment/blob/main/Pakistan_Womens_Employment_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

columns = [
    "Weights", "Province", "Region",
    "S4C5", "S4C6", "S4C7", "S4C9",
    "S5C1", "S5C2", "S5C3", "S5C4",
    "S5C8", "S5C10", "S5C11"
]

df = pd.read_stata(
    "/content/LFS 2024-25.sav web.dta",
    columns=columns,
    convert_categoricals=False
)

# Women aged 15–64
women = df.loc[
    (df["S4C5"] == 2) &
    (df["S4C6"].between(15, 64))
].copy()

print("Women aged 15–64:", len(women))

print("\nEducation categories:")
print(women["S4C9"].value_counts(dropna=False).sort_index())

print("\nEmployment-status categories:")
print(women["S5C11"].value_counts(dropna=False).sort_index())

print("\nRural/urban distribution:")
print(women["Region"].value_counts(dropna=False))

ValueError: buffer is smaller than requested size

In [2]:
import pandas as pd

file_path = "/content/LFS 2024-25.sav web.dta"

columns = [
    "Weights", "Province", "Region",
    "S4C5", "S4C6", "S4C7", "S4C9",
    "S5C1", "S5C2", "S5C3", "S5C4",
    "S5C8", "S5C10", "S5C11"
]

reader = pd.read_stata(
    file_path,
    columns=columns,
    chunksize=10000,
    convert_categoricals=False
)

female_chunks = []

for chunk in reader:
    selected = chunk.loc[
        (chunk["S4C5"] == 2) &
        (chunk["S4C6"].between(15, 64))
    ].copy()

    female_chunks.append(selected)

women = pd.concat(female_chunks, ignore_index=True)

print("Women aged 15–64:", len(women))

print("\nEducation categories:")
print(women["S4C9"].value_counts(dropna=False).sort_index())

print("\nEmployment-status categories:")
print(women["S5C11"].value_counts(dropna=False).sort_index())

print("\nRural/urban distribution:")
print(women["Region"].value_counts(dropna=False))

Women aged 15–64: 89239

Education categories:
S4C9
1.0     43468
2.0       612
3.0      2977
4.0      9269
5.0     10502
6.0     10723
7.0      6557
8.0       118
9.0       190
10.0       96
11.0        2
12.0     2165
13.0      793
14.0     1620
15.0      137
16.0       10
Name: count, dtype: int64

Employment-status categories:
S5C11
1.0       28
2.0     5402
3.0     5323
4.0    11270
5.0       33
6.0       69
NaN    67114
Name: count, dtype: int64

Rural/urban distribution:
Region
1.0    59287
2.0    29952
Name: count, dtype: int64


In [3]:
screening_vars = [
    "S5C1",
    "S5C2",
    "S5C3",
    "S5C4",
    "S5C8",
    "S5C10",
    "S5C11"
]

for var in screening_vars:
    print(f"\n{var}")
    print(
        women[var]
        .value_counts(dropna=False)
        .sort_index()
    )


S5C1
S5C1
1.0     5356
2.0    83883
Name: count, dtype: int64

S5C2
S5C2
1.0     6063
2.0    77820
NaN     5356
Name: count, dtype: int64

S5C3
S5C3
1.0    11623
2.0    66197
NaN    11419
Name: count, dtype: int64

S5C4
S5C4
1.0       28
2.0    66169
NaN    23042
Name: count, dtype: int64

S5C8
S5C8
0.0      327
1.0      216
2.0      592
3.0       87
4.0    65300
NaN    22717
Name: count, dtype: int64

S5C10
S5C10
1.0      239
2.0    12279
3.0      996
4.0      818
NaN    74907
Name: count, dtype: int64

S5C11
S5C11
1.0       28
2.0     5402
3.0     5323
4.0    11270
5.0       33
6.0       69
NaN    67114
Name: count, dtype: int64


In [4]:
import pandas as pd

women["employment_candidate"] = women["S5C11"].isin(
    [1, 2, 3, 4, 5, 6]
).astype(int)

print("Preliminary employment counts:")
print(women["employment_candidate"].value_counts())

print("\nS5C10 by employment candidate:")
print(pd.crosstab(
    women["S5C10"],
    women["employment_candidate"],
    dropna=False
))

print("\nS5C8 by employment candidate:")
print(pd.crosstab(
    women["S5C8"],
    women["employment_candidate"],
    dropna=False
))

Preliminary employment counts:
employment_candidate
0    67114
1    22125
Name: count, dtype: int64

S5C10 by employment candidate:
employment_candidate      0      1
S5C10                             
1.0                       0    239
2.0                       0  12279
3.0                     996      0
4.0                     818      0
NaN                   65300   9607

S5C8 by employment candidate:
employment_candidate      0      1
S5C8                              
0.0                       0    327
1.0                     212      4
2.0                     588      4
3.0                      87      0
4.0                   65300      0
NaN                     927  21790


In [5]:
# Inspect employment-screening patterns
check_vars = [
    "S5C1", "S5C2", "S5C3", "S5C4",
    "S5C8", "S5C10", "S5C11"
]

# All combinations of screening responses
patterns = (
    women.groupby(
        check_vars,
        dropna=False
    )
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

print(patterns.head(25).to_string(index=False))

# Examine employed women with unusual S5C8 responses
unusual = women.loc[
    (women["employment_candidate"] == 1) &
    (women["S5C8"].isin([1, 2])),
    check_vars
]

print("\nUnusual employment records:")
print(unusual.to_string(index=False))

 S5C1  S5C2  S5C3  S5C4  S5C8  S5C10  S5C11  count
  2.0   2.0   2.0   2.0   4.0    NaN    NaN  65281
  2.0   2.0   1.0   NaN   NaN    2.0    4.0   9967
  1.0   NaN   NaN   NaN   NaN    NaN    3.0   5311
  2.0   1.0   NaN   NaN   NaN    NaN    2.0   3248
  2.0   1.0   NaN   NaN   NaN    2.0    2.0   2016
  2.0   2.0   1.0   NaN   NaN    NaN    4.0    904
  2.0   1.0   NaN   NaN   NaN    3.0    NaN    348
  2.0   2.0   2.0   2.0   2.0    3.0    NaN    309
  2.0   2.0   2.0   2.0   2.0    4.0    NaN    275
  2.0   1.0   NaN   NaN   NaN    4.0    NaN    219
  2.0   2.0   1.0   NaN   0.0    2.0    4.0    213
  2.0   2.0   1.0   NaN   NaN    3.0    NaN    201
  2.0   2.0   1.0   NaN   NaN    4.0    NaN    159
  2.0   2.0   1.0   NaN   NaN    1.0    4.0    153
  2.0   2.0   2.0   2.0   1.0    4.0    NaN    142
  2.0   1.0   NaN   NaN   0.0    2.0    2.0     76
  2.0   1.0   NaN   NaN   NaN    NaN    6.0     69
  2.0   2.0   2.0   2.0   1.0    3.0    NaN     68
  2.0   2.0   2.0   2.0   3.0  

In [6]:
import numpy as np
import pandas as pd

# Provisional employment indicator
women["employed"] = women["S5C11"].isin(
    [1, 2, 3, 4, 5, 6]
).astype(int)

# Employment by original education codes
employment_by_education = (
    women.groupby("S4C9", observed=True)
    .agg(
        observations=("employed", "size"),
        employed=("employed", "sum"),
        employment_rate=("employed", "mean")
    )
)

employment_by_education["employment_rate"] *= 100

print(employment_by_education.round(2))

      observations  employed  employment_rate
S4C9                                         
1.0          43468     12436            28.61
2.0            612       155            25.33
3.0           2977       723            24.29
4.0           9269      2257            24.35
5.0          10502      1715            16.33
6.0          10723      1717            16.01
7.0           6557      1314            20.04
8.0            118        50            42.37
9.0            190        94            49.47
10.0            96        24            25.00
11.0             2         2           100.00
12.0          2165       635            29.33
13.0           793       244            30.77
14.0          1620       692            42.72
15.0           137        62            45.26
16.0            10         5            50.00


In [7]:
import numpy as np
import pandas as pd

education_map = {
    1: "No formal schooling",
    2: "Below middle",
    3: "Below middle",
    4: "Below middle",
    5: "Middle to below intermediate",
    6: "Middle to below intermediate",
    7: "Intermediate to below degree",
    8: "Degree and above",
    9: "Degree and above",
    10: "Degree and above",
    11: "Degree and above",
    12: "Degree and above",
    13: "Degree and above",
    14: "Degree and above",
    15: "Degree and above",
    16: "Degree and above"
}

women["education_group"] = women["S4C9"].map(education_map)

# Calculate survey-weighted employment rates
weighted_rates = (
    women.groupby("education_group")
    .apply(
        lambda x: pd.Series({
            "sample_size": len(x),
            "employment_rate": (
                np.average(
                    x["employed"],
                    weights=x["Weights"]
                ) * 100
            )
        }),
        include_groups=False
    )
)

print(weighted_rates.round(2))

                              sample_size  employment_rate
education_group                                           
Below middle                      12858.0            24.37
Degree and above                   5131.0            32.07
Intermediate to below degree       6557.0            19.41
Middle to below intermediate      21225.0            15.88
No formal schooling               43468.0            29.55


In [8]:
import pandas as pd
import numpy as np

employment_labels = {
    1: "Employer",
    2: "Independent worker",
    3: "Employee",
    4: "Contributing family worker",
    5: "Paid apprentice/intern",
    6: "Dependent contractor"
}

employed_women = women.loc[
    women["S5C11"].isin(employment_labels)
].copy()

employed_women["employment_type"] = (
    employed_women["S5C11"].map(employment_labels)
)

# Weighted employment composition within each education group
composition = pd.crosstab(
    employed_women["education_group"],
    employed_women["employment_type"],
    values=employed_women["Weights"],
    aggfunc="sum",
    normalize="index"
) * 100

print(composition.round(2).to_string())

employment_type               Contributing family worker  Dependent contractor  Employee  Employer  Independent worker  Paid apprentice/intern
education_group                                                                                                                               
Below middle                                       51.01                  0.31     19.36      0.00               29.17                    0.15
Degree and above                                   15.01                  0.18     63.80      0.65               20.31                    0.05
Intermediate to below degree                       23.64                  0.65     50.74      0.42               24.06                    0.49
Middle to below intermediate                       43.06                  0.51     23.12      0.19               32.69                    0.43
No formal schooling                                58.59                  0.35     18.13      0.03               22.83                    0.06

In [9]:
import numpy as np
import pandas as pd

women["area"] = women["Region"].map({
    1: "Rural",
    2: "Urban"
})

# Weighted employment rate by education and area
def weighted_rate(group):
    return 100 * np.average(
        group["employed"],
        weights=group["Weights"]
    )

area_employment = (
    women.groupby(
        ["education_group", "area"],
        observed=True
    )
    .apply(weighted_rate, include_groups=False)
    .unstack()
)

print("Weighted employment rates (%):")
print(area_employment.round(2))

# Among employed women: weighted share in family work
family_work = employed_women.copy()

family_work["area"] = family_work["Region"].map({
    1: "Rural",
    2: "Urban"
})

family_work["family_worker"] = (
    family_work["S5C11"] == 4
).astype(int)

family_share = (
    family_work.groupby(
        ["education_group", "area"],
        observed=True
    )
    .apply(
        lambda g: 100 * np.average(
            g["family_worker"],
            weights=g["Weights"]
        ),
        include_groups=False
    )
    .unstack()
)

print("\nContributing family workers among employed women (%):")
print(family_share.round(2))


Weighted employment rates (%):
area                          Rural  Urban
education_group                           
Below middle                  33.50  11.01
Degree and above              45.56  24.94
Intermediate to below degree  29.28  12.93
Middle to below intermediate  23.84   8.85
No formal schooling           35.10  13.28

Contributing family workers among employed women (%):
area                          Rural  Urban
education_group                           
Below middle                  55.25  32.16
Degree and above              15.64  14.40
Intermediate to below degree  25.69  20.58
Middle to below intermediate  49.26  28.33
No formal schooling           61.20  38.35


In [10]:
import numpy as np
import pandas as pd

marital_labels = {
    1: "Never married",
    2: "Married",
    3: "Widowed",
    4: "Divorced"
}

women["marital_status"] = women["S4C7"].map(marital_labels)

def weighted_employment(group):
    valid = group.dropna(subset=["Weights", "employed"])
    return 100 * np.average(
        valid["employed"],
        weights=valid["Weights"]
    )

marital_employment = (
    women.dropna(subset=["marital_status"])
    .groupby(
        ["education_group", "marital_status"],
        observed=True
    )
    .apply(weighted_employment, include_groups=False)
    .unstack()
)

print("Employment rates by education and marital status (%):")
print(marital_employment.round(2))

print("\nUnweighted cell counts:")
print(pd.crosstab(
    women["education_group"],
    women["marital_status"]
))

Employment rates by education and marital status (%):
marital_status                Divorced  Married  Never married  Widowed
education_group                                                        
Below middle                     33.62    24.53          24.55    20.00
Degree and above                 31.76    31.36          33.11    39.20
Intermediate to below degree     27.65    21.47          17.80    24.10
Middle to below intermediate     23.26    20.46           9.94    18.29
No formal schooling              40.56    29.95          29.09    24.90

Unweighted cell counts:
marital_status                Divorced  Married  Never married  Widowed
education_group                                                        
Below middle                        41     9467           2827      523
Degree and above                    27     3277           1757       70
Intermediate to below degree        19     2670           3802       66
Middle to below intermediate        73    11281          

In [11]:
import numpy as np
import pandas as pd

# First inspect the province codes
print("Province codes:")
print(
    women["Province"]
    .value_counts(dropna=False)
    .sort_index()
)

# Calculate weighted employment rates by province
province_rates = (
    women.groupby("Province", observed=True)
    .apply(
        lambda g: pd.Series({
            "sample_size": len(g),
            "employment_rate": 100 * np.average(
                g["employed"],
                weights=g["Weights"]
            )
        }),
        include_groups=False
    )
)

print("\nFemale employment by province:")
print(
    province_rates
    .sort_values("employment_rate")
    .round(2)
)

Province codes:
Province
1.0    17653
2.0    40321
3.0    21311
4.0     9954
Name: count, dtype: int64

Female employment by province:
          sample_size  employment_rate
Province                              
4.0            9954.0            13.05
1.0           17653.0            17.19
3.0           21311.0            17.43
2.0           40321.0            30.70


In [12]:
import numpy as np
import pandas as pd

province_map = {
    1: "Khyber Pakhtunkhwa",
    2: "Punjab",
    3: "Sindh",
    4: "Balochistan"
}

women["province_name"] = women["Province"].map(province_map)

# Weighted employment rates by province and education
province_education = (
    women.groupby(
        ["province_name", "education_group"],
        observed=True
    )
    .apply(
        lambda g: pd.Series({
            "sample_size": len(g),
            "employment_rate": 100 * np.average(
                g["employed"],
                weights=g["Weights"]
            )
        }),
        include_groups=False
    )
    .reset_index()
)

rates = province_education.pivot(
    index="education_group",
    columns="province_name",
    values="employment_rate"
)

counts = province_education.pivot(
    index="education_group",
    columns="province_name",
    values="sample_size"
)

print("Weighted employment rates (%):")
print(rates.round(2).to_string())

print("\nUnweighted sample sizes:")
print(counts.to_string())

Weighted employment rates (%):
province_name                 Balochistan  Khyber Pakhtunkhwa  Punjab  Sindh
education_group                                                             
Below middle                         6.29               12.45   32.38  11.57
Degree and above                    54.20               43.13   31.46  24.56
Intermediate to below degree        10.97               11.72   23.62  11.97
Middle to below intermediate         4.71                8.73   20.65   7.22
No formal schooling                 13.91               19.33   40.16  25.16

Unweighted sample sizes:
province_name                 Balochistan  Khyber Pakhtunkhwa   Punjab    Sindh
education_group                                                                
Below middle                        699.0              1771.0   6945.0   3443.0
Degree and above                    187.0               917.0   3121.0    906.0
Intermediate to below degree        298.0              1138.0   3587.0   1534.0
Midd

In [13]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive
